In [1]:
!nvidia-smi

Fri Mar 27 19:19:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!git clone https://github.com/davelee-uestc/nsf_debiasing.git
%cd nsf_debiasing

Cloning into 'nsf_debiasing'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 61 (delta 3), reused 53 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (61/61), 77.70 KiB | 5.55 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/nsf_debiasing


In [3]:
# PyTorch is already installed in Colab, so skip that.
# Install the remaining packages:

!pip install timm transformers scikit-learn tqdm wilds -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 8.2 MB/s eta 0:00:00


In [4]:
!pip install opencv-python-headless -q
!pip install vissl -f https://dl.fbaipublicfiles.com/vissl/packaging/apexwheels/index.html -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.5/47.5 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 88.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
ERROR: Exception:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 377, in run
    requirement_set = resolver.resolve(
                      ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/resolution/resolvelib/resolver.py", line 95, in resolve
    resu

In [6]:
import torch
import timm
import transformers
import sklearn
import wilds
print("✅ PyTorch:", torch.__version__)
print("✅ CUDA available:", torch.cuda.is_available())
print("✅ All packages imported successfully!")

✅ PyTorch: 2.10.0+cu128
✅ CUDA available: True
✅ All packages imported successfully!


In [7]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/nsf_results', exist_ok=True)
print("✅ Drive mounted and folder created!")

Mounted at /content/drive
✅ Drive mounted and folder created!


In [8]:
# Make the script executable
!chmod +x train_sup.sh

# Run training for CivilComments (auto-downloads)
!./train_sup.sh -d cc

All idle GPUs: ,0
Running script with the following parameters:
Dataset: cc
Batch size: 16
Number of epochs: 10
Seed: 1
GPU: 0


In [9]:
# Match whichever dataset you trained above
!python3 ssc.py civilcomments   # if you used -d cc
# !python3 ssc.py waterbirds    # if you used -d wb
# !python3 ssc.py celeba        # if you used -d ce
# !python3 ssc.py multinli      # if you used -d mn

2026-03-27 19:27:23.096415: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774639643.116660   10747 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774639643.123393   10747 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774639643.140935   10747 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774639643.140961   10747 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774639643.140964   10747 computation_placer.cc:177] computation placer alr

In [10]:
!pip install mmcv -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 479.1/479.1 kB 13.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 452.7/452.7 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 30.2 MB/s eta 0:00:00
ERROR: Operation cancelled by user


In [11]:
!pip install mmcv-lite -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 732.3/732.3 kB 16.7 MB/s eta 0:00:00


In [12]:
!grep -n "mmcv" /content/nsf_debiasing/ssc_common.py

4:import mmcv
190:    prog=mmcv.ProgressBar(len(loader))
217:    prog=mmcv.ProgressBar(N)
246:    prog=mmcv.ProgressBar(N)


In [13]:
# Read the file
with open('/content/nsf_debiasing/ssc_common.py', 'r') as f:
    content = f.read()

# Replace mmcv import with tqdm
content = content.replace(
    'import mmcv',
    'from tqdm import tqdm'
)

# Replace all 3 uses of mmcv.ProgressBar
content = content.replace(
    'prog=mmcv.ProgressBar(len(loader))',
    'prog=tqdm(total=len(loader))'
)
content = content.replace(
    'prog=mmcv.ProgressBar(N)',
    'prog=tqdm(total=N)'
)

# Write it back
with open('/content/nsf_debiasing/ssc_common.py', 'w') as f:
    f.write(content)

print("✅ Patched successfully!")

✅ Patched successfully!


In [14]:
!grep -n "mmcv\|tqdm\|prog=" /content/nsf_debiasing/ssc_common.py

4:from tqdm import tqdm
190:    prog=tqdm(total=len(loader))
217:    prog=tqdm(total=N)
246:    prog=tqdm(total=N)


In [15]:
!grep -n "prog\." /content/nsf_debiasing/ssc_common.py

205:        prog.update(1)
227:        prog.update(1)
257:        prog.update(1)


In [16]:
!chmod +x train_sup.sh
!./train_sup.sh -d cc

All idle GPUs: ,0
Running script with the following parameters:
Dataset: cc
Batch size: 16
Number of epochs: 10
Seed: 1
GPU: 0


In [17]:
!python3 ssc.py civilcomments

2026-03-27 19:47:10.791870: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774640830.812509   16262 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774640830.819152   16262 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774640830.836120   16262 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774640830.836174   16262 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774640830.836185   16262 computation_placer.cc:177] computation placer alr

In [18]:
import os
print(os.path.exists('/content/nsf_debiasing/logs/civilcomments/erm_seed1/final_checkpoint.pt'))
!ls /content/nsf_debiasing/logs/ 2>/dev/null || echo "❌ logs folder doesn't exist yet!"

False
❌ logs folder doesn't exist yet!


In [19]:
import os
os.chdir('/content/nsf_debiasing')
!pwd  # should print /content/nsf_debiasing

/content/nsf_debiasing


In [20]:
!cat train_sup.sh

#!/bin/bash
nvidia_smi_output=$(nvidia-smi|grep MiB |egrep -o '[0-9]+MiB /'|egrep -o [0-9]+)
idle_gpu_id=-1
idx=-1
idle_gpu_ids=""

while read -r line; do
    idx=$(($idx + 1))
    #if [[ $line == *" 0% "* ]] && [[ $line == *" 8MiB "* ]]; then
    if [ $line -lt 100 ] ;then
        idle_gpu_id=$idx
        idle_gpu_ids=("$idle_gpu_ids,$idx")
    fi
done <<< "$nvidia_smi_output"

if [ $idle_gpu_id -ge 0 ]; then
    echo "All idle GPUs: $idle_gpu_ids"
else
    echo "No idle GPUs found."
    exit 1
fi

#!/bin/bash

# Function to display usage
usage() {
    echo "Usage: $0 -d <dataset> [-b <batchsize>] [-n <n_epoch>] [-s <seed>]"
    echo "  -d <dataset>   : Required. Dataset name. Options are 'ce', 'wb', 'mn', 'cc', 'cx'."
    echo "  -b <batchsize> : Optional. Batch size. Default depends on dataset."
    echo "  -n <n_epoch>   : Optional. Number of epochs. Default depends on dataset."
    echo "  -s <seed>      : Optional. Seed for randomness. Default depends on dataset."
    exit 1
}

#

In [21]:
!CUDA_VISIBLE_DEVICES=0 python3 train_supervised.py \
    --output_dir=logs/civilcomments/erm_seed1 \
    --num_epochs=10 \
    --eval_freq=1 \
    --save_freq=10 \
    --seed=1 \
    --weight_decay=1e-4 \
    --batch_size=16 \
    --init_lr=1e-5 \
    --scheduler=bert_lr_scheduler \
    --data_dir=cc \
    --data_transform=BertTokenizeTransform \
    --dataset=WildsCivilCommentsCoarse \
    --model=bert_pretrained \
    --optimizer=bert_adamw_optimizer

2026-03-27 19:51:42.264699: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774641102.287867   17415 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774641102.295442   17415 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774641102.315584   17415 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774641102.315612   17415 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774641102.315616   17415 computation_placer.cc:177] computation placer alr

In [22]:
import os
os.environ['WANDB_DISABLED'] = 'true'

In [25]:
!WANDB_DISABLED=true CUDA_VISIBLE_DEVICES=0 python3 train_supervised.py \
    --output_dir=logs/civilcomments/erm_seed1 \
    --num_epochs=10 \
    --eval_freq=1 \
    --save_freq=10 \
    --seed=1 \
    --weight_decay=1e-4 \
    --batch_size=16 \
    --init_lr=1e-5 \
    --scheduler=bert_lr_scheduler \
    --data_dir=cc \
    --data_transform=BertTokenizeTransform \
    --dataset=WildsCivilCommentsCoarse \
    --model=bert_pretrained \
    --optimizer=bert_adamw_optimizer


2026-03-27 20:38:21.227724: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774643901.248433   29769 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774643901.255347   29769 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774643901.272817   29769 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774643901.272844   29769 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774643901.272847   29769 computation_placer.cc:177] computation placer alr

In [26]:
import shutil, os

# Remove old wandb runs that have saved bad configs
wandb_dir = '/content/nsf_debiasing/logs/civilcomments/erm_seed1/wandb'
if os.path.exists(wandb_dir):
    shutil.rmtree(wandb_dir)
    print("✅ Old wandb config deleted!")
else:
    print("No wandb dir found, that's fine!")

✅ Old wandb config deleted!


In [27]:
with open('/content/nsf_debiasing/train_supervised.py', 'r') as f:
    content = f.read()

# Comment out all wandb-related lines
content = content.replace(
    'if has_wandb:\n        wandb.init(dir=args.output_dir)\n        args.__dict__.update(wandb.config)',
    '# wandb disabled\n        pass'
)

with open('/content/nsf_debiasing/train_supervised.py', 'w') as f:
    f.write(content)

print("✅ wandb patched out!")

✅ wandb patched out!


In [28]:
!grep -n "wandb" /content/nsf_debiasing/train_supervised.py

8:    import wandb
9:    has_wandb = True
11:    has_wandb = False
19:    # wandb disabled


In [29]:
import os
os.chdir('/content/nsf_debiasing')
os.makedirs('/content/nsf_debiasing/cc', exist_ok=True)

!CUDA_VISIBLE_DEVICES=0 python3 train_supervised.py \
    --output_dir=/content/nsf_debiasing/logs/civilcomments/erm_seed1 \
    --num_epochs=10 \
    --eval_freq=1 \
    --save_freq=10 \
    --seed=1 \
    --weight_decay=1e-4 \
    --batch_size=16 \
    --init_lr=1e-5 \
    --scheduler=bert_lr_scheduler \
    --data_dir=/content/nsf_debiasing/cc \
    --data_transform=BertTokenizeTransform \
    --dataset=WildsCivilCommentsCoarse \
    --model=bert_pretrained \
    --optimizer=bert_adamw_optimizer

  File "/content/nsf_debiasing/train_supervised.py", line 23
    utils.set_seed(args.seed)
                             ^
IndentationError: unindent does not match any outer indentation level


In [30]:
!git checkout /content/nsf_debiasing/train_supervised.py
print("✅ File restored!")

Updated 1 path from the index
✅ File restored!


In [31]:
with open('/content/nsf_debiasing/train_supervised.py', 'r') as f:
    content = f.read()

# Simply disable wandb by making has_wandb always False
content = content.replace(
    'try:\n    import wandb\n    has_wandb = True\nexcept ImportError:\n    has_wandb = False',
    'has_wandb = False  # wandb disabled'
)

with open('/content/nsf_debiasing/train_supervised.py', 'w') as f:
    f.write(content)

print("✅ Patched cleanly!")

✅ Patched cleanly!


In [33]:
!grep -n "wandb\|has_wandb" /content/nsf_debiasing/train_supervised.py


8:    import wandb
9:    has_wandb = True
11:    has_wandb = False
19:    if has_wandb:
20:        wandb.init(dir=args.output_dir)
21:        args.__dict__.update(wandb.config)


In [34]:
import os
os.chdir('/content/nsf_debiasing')
os.makedirs('/content/nsf_debiasing/cc', exist_ok=True)

!CUDA_VISIBLE_DEVICES=0 python3 train_supervised.py \
    --output_dir=/content/nsf_debiasing/logs/civilcomments/erm_seed1 \
    --num_epochs=10 \
    --eval_freq=1 \
    --save_freq=10 \
    --seed=1 \
    --weight_decay=1e-4 \
    --batch_size=16 \
    --init_lr=1e-5 \
    --scheduler=bert_lr_scheduler \
    --data_dir=/content/nsf_debiasing/cc \
    --data_transform=BertTokenizeTransform \
    --dataset=WildsCivilCommentsCoarse \
    --model=bert_pretrained \
    --optimizer=bert_adamw_optimizer

2026-03-27 20:50:40.940611: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774644640.961958   32921 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774644640.968988   32921 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774644640.987480   32921 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774644640.987506   32921 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774644640.987510   32921 computation_placer.cc:177] computation placer alr

In [35]:
import wilds
dataset = wilds.get_dataset(dataset='civilcomments', root_dir='/content/nsf_debiasing/cc', download=True)
print("✅ Dataset downloaded!")

You can also download the dataset manually at https://wilds.stanford.edu/downloads.


90914816Byte [00:06, 13913916.44Byte/s]                            


Extracting /content/nsf_debiasing/cc/civilcomments_v1.0/archive.tar.gz to /content/nsf_debiasing/cc/civilcomments_v1.0

It took 0.14 minutes to download and uncompress the dataset.

✅ Dataset downloaded!


In [36]:
!CUDA_VISIBLE_DEVICES=0 python3 train_supervised.py \
    --output_dir=/content/nsf_debiasing/logs/civilcomments/erm_seed1 \
    --num_epochs=10 \
    --eval_freq=1 \
    --save_freq=10 \
    --seed=1 \
    --weight_decay=1e-4 \
    --batch_size=16 \
    --init_lr=1e-5 \
    --scheduler=bert_lr_scheduler \
    --data_dir=/content/nsf_debiasing/cc \
    --data_transform=BertTokenizeTransform \
    --dataset=WildsCivilCommentsCoarse \
    --model=bert_pretrained \
    --optimizer=bert_adamw_optimizer

2026-03-27 20:52:35.082995: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774644755.103889   33455 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774644755.110940   33455 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774644755.129232   33455 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774644755.129259   33455 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774644755.129262   33455 computation_placer.cc:177] computation placer alr

In [37]:
import os
os.chdir('/content/nsf_debiasing')
os.makedirs('/content/nsf_debiasing/cifar10', exist_ok=True)

In [38]:
!CUDA_VISIBLE_DEVICES=0 python3 train_supervised.py \
    --output_dir=/content/nsf_debiasing/logs/cifar10/erm_seed1 \
    --num_epochs=20 \
    --eval_freq=1 \
    --save_freq=10 \
    --seed=1 \
    --weight_decay=1e-4 \
    --batch_size=100 \
    --init_lr=3e-3 \
    --scheduler=cosine_lr_scheduler \
    --data_dir=/content/nsf_debiasing/cifar10 \
    --dataset=FakeSpuriousCIFAR10 \
    --model=imagenet_resnet50_pretrained

2026-03-27 21:01:05.344173: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774645265.364961   35634 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774645265.371817   35634 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774645265.389637   35634 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774645265.389673   35634 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774645265.389677   35634 computation_placer.cc:177] computation placer alr

In [39]:
!CUDA_VISIBLE_DEVICES=0 python3 train_supervised.py \
    --output_dir=/content/nsf_debiasing/logs/cifar10/erm_seed1 \
    --num_epochs=10 \
    --eval_freq=1 \
    --save_freq=10 \
    --seed=1 \
    --weight_decay=1e-4 \
    --batch_size=100 \
    --init_lr=3e-3 \
    --scheduler=cosine_lr_scheduler \
    --data_dir=/content/nsf_debiasing/cifar10 \
    --dataset=FakeSpuriousCIFAR10 \
    --model=imagenet_resnet50_pretrained

2026-03-27 21:23:53.271890: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774646633.293373   42088 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774646633.300724   42088 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774646633.320329   42088 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774646633.320358   42088 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774646633.320363   42088 computation_placer.cc:177] computation placer alr

In [40]:
!cp -r /content/nsf_debiasing/logs /content/drive/MyDrive/nsf_results/cifar10_logs
print("✅ CIFAR-10 checkpoint saved!")

✅ CIFAR-10 checkpoint saved!


In [41]:
!python3 ssc.py cifar10

2026-03-27 23:01:33.259102: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774652493.280322   69277 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774652493.287482   69277 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774652493.306715   69277 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774652493.306775   69277 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774652493.306780   69277 computation_placer.cc:177] computation placer alr

In [42]:
!grep -n "waterbirds\|celeba\|cifar\|multinli\|civilcomments\|chexpert" /content/nsf_debiasing/ssc.py | head -30


105:        # "chexpert":['--data_dir=chexpert',
107:        #           '--model=imagenet_resnet50_pretrained', '--ckpt_path=logs/chexpert/erm_seed1/final_checkpoint.pt',
109:        "waterbirds":['--data_dir=waterbirds',
111:                  '--model=imagenet_resnet50_pretrained', '--ckpt_path=logs/waterbirds/erm_seed1/final_checkpoint.pt',
113:        "celeba":['--data_dir=celeba',
115:                  '--model=imagenet_resnet50_pretrained', '--ckpt_path=logs/celeba/erm_seed1/final_checkpoint.pt',
117:        "multinli":[
118:            '--data_dir=multinli',
123:            '--ckpt_path=logs/multinli/erm_seed1/final_checkpoint.pt', '--num_epochs=1200', '--num_epochs_ft=1500'],
124:        "civilcomments":['--data_dir=cc',
127:                  '--ckpt_path=logs/civilcomments/erm_seed1/final_checkpoint.pt', '--num_epochs=300'],


In [43]:
!sed -n '100,140p' /content/nsf_debiasing/ssc.py

    return main_exp(all_train_data, all_test_data, n_classes, logger, model=model, N1=args.num_epochs, N2=args.num_epochs_ft)


if __name__ == '__main__':
    all_args_map={
        # "chexpert":['--data_dir=chexpert',
        #           '--data_transform=AugWaterbirdsCelebATransform', '--dataset=SpuriousCorrelationDataset',
        #           '--model=imagenet_resnet50_pretrained', '--ckpt_path=logs/chexpert/erm_seed1/final_checkpoint.pt',
        #           '--label_filename=metadata.csv', '--batch_size=64', '--num_epochs=650', '--num_epochs_ft=10000'],
        "waterbirds":['--data_dir=waterbirds',
                  '--data_transform=AugWaterbirdsCelebATransform', '--dataset=SpuriousCorrelationDataset',
                  '--model=imagenet_resnet50_pretrained', '--ckpt_path=logs/waterbirds/erm_seed1/final_checkpoint.pt',
                  '--label_filename=metadata.csv', '--batch_size=64', '--num_epochs=10', '--num_epochs_ft=500'],
        "celeba":['--data_dir=celeba',
          

In [44]:
with open('/content/nsf_debiasing/ssc.py', 'r') as f:
    content = f.read()

# Add cifar10 entry after the civilcomments entry
cifar10_entry = '''        "cifar10":['--data_dir=/content/nsf_debiasing/cifar10',
                  '--data_transform=AugWaterbirdsCelebATransform', '--dataset=FakeSpuriousCIFAR10',
                  '--model=imagenet_resnet50_pretrained', '--ckpt_path=logs/cifar10/erm_seed1/final_checkpoint.pt',
                  '--batch_size=64', '--num_epochs=10', '--num_epochs_ft=500'],'''

content = content.replace(
    '"civilcomments":[\'--data_dir=cc\',',
    cifar10_entry + '\n        "civilcomments":[\'--data_dir=cc\','
)

with open('/content/nsf_debiasing/ssc.py', 'w') as f:
    f.write(content)

print("✅ cifar10 added to ssc.py!")

✅ cifar10 added to ssc.py!


In [45]:
!sed -n '100,145p' /content/nsf_debiasing/ssc.py

    return main_exp(all_train_data, all_test_data, n_classes, logger, model=model, N1=args.num_epochs, N2=args.num_epochs_ft)


if __name__ == '__main__':
    all_args_map={
        # "chexpert":['--data_dir=chexpert',
        #           '--data_transform=AugWaterbirdsCelebATransform', '--dataset=SpuriousCorrelationDataset',
        #           '--model=imagenet_resnet50_pretrained', '--ckpt_path=logs/chexpert/erm_seed1/final_checkpoint.pt',
        #           '--label_filename=metadata.csv', '--batch_size=64', '--num_epochs=650', '--num_epochs_ft=10000'],
        "waterbirds":['--data_dir=waterbirds',
                  '--data_transform=AugWaterbirdsCelebATransform', '--dataset=SpuriousCorrelationDataset',
                  '--model=imagenet_resnet50_pretrained', '--ckpt_path=logs/waterbirds/erm_seed1/final_checkpoint.pt',
                  '--label_filename=metadata.csv', '--batch_size=64', '--num_epochs=10', '--num_epochs_ft=500'],
        "celeba":['--data_dir=celeba',
          

In [46]:
!python3 ssc.py cifar10

2026-03-27 23:08:38.047190: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774652918.068365   71084 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774652918.075430   71084 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774652918.093365   71084 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774652918.093389   71084 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774652918.093393   71084 computation_placer.cc:177] computation placer alr

In [47]:
with open('/content/nsf_debiasing/ssc.py', 'r') as f:
    content = f.read()

content = content.replace(
    '"cifar10":[\'--data_dir=/content/nsf_debiasing/cifar10\',\n                  \'--data_transform=AugWaterbirdsCelebATransform\', \'--dataset=FakeSpuriousCIFAR10\',\n                  \'--model=imagenet_resnet50_pretrained\', \'--ckpt_path=logs/cifar10/erm_seed1/final_checkpoint.pt\',\n                  \'--batch_size=64\', \'--num_epochs=10\', \'--num_epochs_ft=500\'],',
    '"cifar10":[\'--data_dir=/content/nsf_debiasing/cifar10\',\n                  \'--data_transform=AugWaterbirdsCelebATransform\', \'--dataset=FakeSpuriousCIFAR10\',\n                  \'--model=imagenet_resnet50_pretrained\', \'--ckpt_path=logs/cifar10/erm_seed1/final_checkpoint.pt\',\n                  \'--batch_size=64\', \'--n_classes=10\', \'--num_epochs=10\', \'--num_epochs_ft=500\'],'
)

with open('/content/nsf_debiasing/ssc.py', 'w') as f:
    f.write(content)

print("✅ Fixed n_classes=10 for cifar10!")

✅ Fixed n_classes=10 for cifar10!


In [48]:
!grep -A5 "cifar10" /content/nsf_debiasing/ssc.py

                "cifar10":['--data_dir=/content/nsf_debiasing/cifar10',
                  '--data_transform=AugWaterbirdsCelebATransform', '--dataset=FakeSpuriousCIFAR10',
                  '--model=imagenet_resnet50_pretrained', '--ckpt_path=logs/cifar10/erm_seed1/final_checkpoint.pt',
                  '--batch_size=64', '--n_classes=10', '--num_epochs=10', '--num_epochs_ft=500'],
        "civilcomments":['--data_dir=cc',
                  '--data_transform=BertTokenizeTransform', '--dataset=WildsCivilCommentsCoarse',
                  '--model=bert_pretrained', '--batch_size=64',
                  '--ckpt_path=logs/civilcomments/erm_seed1/final_checkpoint.pt', '--num_epochs=300'],


In [49]:
!python3 ssc.py cifar10

2026-03-27 23:10:30.981088: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774653031.001967   71577 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774653031.008872   71577 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774653031.027059   71577 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774653031.027092   71577 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774653031.027096   71577 computation_placer.cc:177] computation placer alr

In [50]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

# Create a master folder for all results
os.makedirs('/content/drive/MyDrive/nsf_results', exist_ok=True)

# Save logs (checkpoints + results)
!cp -r /content/nsf_debiasing/logs /content/drive/MyDrive/nsf_results/

# Save the modified ssc.py (with cifar10 added)
!cp /content/nsf_debiasing/ssc.py /content/drive/MyDrive/nsf_results/

# Save the modified ssc_common.py (mmcv fix)
!cp /content/nsf_debiasing/ssc_common.py /content/drive/MyDrive/nsf_results/

# Save the modified train_supervised.py (wandb fix)
!cp /content/nsf_debiasing/train_supervised.py /content/drive/MyDrive/nsf_results/

print("✅ Everything saved!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Everything saved!


In [52]:
!ls /content/drive/MyDrive/nsf_results/
!ls /content/drive/MyDrive/nsf_results/logs/


cifar10_logs  logs  ssc_common.py  ssc.py  train_supervised.py
cifar10  civilcomments


In [53]:
from google.colab import files

# Zip all logs + modified files together
!zip -r /content/nsf_results.zip \
    /content/nsf_debiasing/logs \
    /content/nsf_debiasing/ssc.py \
    /content/nsf_debiasing/ssc_common.py \
    /content/nsf_debiasing/train_supervised.py

# Download the zip
files.download('/content/nsf_results.zip')

  adding: content/nsf_debiasing/logs/ (stored 0%)
  adding: content/nsf_debiasing/logs/civilcomments/ (stored 0%)
  adding: content/nsf_debiasing/logs/civilcomments/erm_seed1/ (stored 0%)
  adding: content/nsf_debiasing/logs/civilcomments/erm_seed1/wandb/ (stored 0%)
  adding: content/nsf_debiasing/logs/civilcomments/erm_seed1/wandb/debug.log (deflated 71%)
  adding: content/nsf_debiasing/logs/civilcomments/erm_seed1/wandb/run-20260327_205045-j2qo8ifm/ (stored 0%)
  adding: content/nsf_debiasing/logs/civilcomments/erm_seed1/wandb/run-20260327_205045-j2qo8ifm/files/ (stored 0%)
  adding: content/nsf_debiasing/logs/civilcomments/erm_seed1/wandb/run-20260327_205045-j2qo8ifm/files/wandb-metadata.json (deflated 51%)
  adding: content/nsf_debiasing/logs/civilcomments/erm_seed1/wandb/run-20260327_205045-j2qo8ifm/files/output.log (deflated 64%)
  adding: content/nsf_debiasing/logs/civilcomments/erm_seed1/wandb/run-20260327_205045-j2qo8ifm/files/wandb-summary.json (deflated 19%)
  adding: conte

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>